# Debugging toolkit for research notebooks

How to find what is wrong in a notebook you did not write. Every tool is shown first on a
tiny frame you can check by eye, then once on the real hourly data.

**What's in here**
- `trace`: a helper that prints the vital signs of a frame inside a `.pipe` chain
- assertions: what they look like when they pass and when they fail
- `pd.testing` / `np.testing`: compare two frames with a useful diff
- `df.compare`, `merge(indicator=True)`: find where two frames differ
- recompute a feature two ways (vectorised vs a loop) on a few rows
- leakage tests on a 6-row toy, then on real data
- silent failures: things that do not raise but are wrong
- warnings as errors, copy-on-write
- the minimal reproducible example
- the pandas errors you will meet, one 3-row example each
- the six checks when someone says "R² is 0.99"

In [1]:
import warnings
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## 1. `trace`: print the vital signs of a frame

Start with a 3-row frame. The things you want to know at every step: shape, is the
index unique, is it sorted, how many NaN, how many duplicate rows.

In [2]:
df = pd.DataFrame({"x": [1, 2, 2], "y": [10.0, np.nan, 30.0]}, index=[0, 1, 1])
df

,x,y
0,1,10.0
1,2,NaN
1,2,30.0


In [3]:
print("shape      :", df.shape)
print("unique idx :", df.index.is_unique)
print("sorted idx :", df.index.is_monotonic_increasing)
print("nan cells  :", df.isna().sum().sum())
print("dup rows   :", df.duplicated().sum())

shape      : (3, 2)
unique idx : False
sorted idx : True
nan cells  : 1
dup rows   : 0


Wrap those five lines in a function that **returns the frame unchanged**. Then it can sit
inside a `.pipe(...)` chain without altering the result.

In [4]:
def trace(df, label=""):
    print(label, "| shape", df.shape,
          "| unique", df.index.is_unique,
          "| sorted", df.index.is_monotonic_increasing,
          "| nan", df.isna().sum().sum(),
          "| dups", df.duplicated().sum())
    return df

In [5]:
out = (df
       .pipe(trace, "start   ")
       .drop_duplicates()
       .pipe(trace, "dedup   ")
       .dropna()
       .pipe(trace, "dropna  "))
out

start    | shape (3, 2) | unique False | sorted True | nan 1 | dups 0
dedup    | shape (3, 2) | unique False | sorted True | nan 1 | dups 0
dropna   | shape (2, 2) | unique True | sorted True | nan 0 | dups 0


,x,y
0,1,10.0
1,2,30.0


Each line is the state after one step. `drop_duplicates` removed one of the two rows
labelled 1 (they were identical), `dropna` removed the row with NaN.

**Pitfall:** `drop_duplicates()` only removes rows that are identical in every column. If
the same timestamp appears twice with *different* values, both stay and the index is still
not unique.

In [6]:
d = pd.DataFrame({"time": ["01:00", "02:00", "02:00"], "value": [10, 20, 21]}).set_index("time")
d

,value
time,
01:00,10
02:00,20
02:00,21


In [7]:
after_row_dedup = d.drop_duplicates()
print("rows:", len(after_row_dedup), "| index unique:", after_row_dedup.index.is_unique)
after_row_dedup

rows: 3 | index unique: False


,value
time,
01:00,10
02:00,20
02:00,21


In [8]:
after_key_dedup = d[~d.index.duplicated(keep="last")]
print("rows:", len(after_key_dedup), "| index unique:", after_key_dedup.index.is_unique)
after_key_dedup

rows: 2 | index unique: True


,value
time,
01:00,10
02:00,21


Row dedup kept both `02:00` rows (values 20 and 21 differ). Key dedup kept the last one.

### On real data

In [9]:
raw = pd.read_csv("../data/hourly_power_raw.csv")
clean = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
raw["time"] = pd.to_datetime(raw["time"], utc=True)

out = (raw
       .pipe(trace, "raw     ")
       .drop_duplicates()
       .pipe(trace, "dedup   ")
       .set_index("time").sort_index()
       .pipe(trace, "indexed "))

raw      | shape (17457, 7) | unique True | sorted True | nan 149 | dups 15
dedup    | shape (17442, 7) | unique True | sorted True | nan 149 | dups 0
indexed  | shape (17442, 6) | unique True | sorted True | nan 149 | dups 0


## 2. Assertions

An `assert` is a one-line statement of what you believe about the data. When it is true
nothing happens. When it is false you get an error right there, not three cells later.

In [10]:
good = pd.DataFrame({"v": [1, 2, 3]}, index=[10, 20, 30])
assert good.index.is_unique
assert good.index.is_monotonic_increasing
print("both assertions passed, nothing printed by assert itself")

both assertions passed, nothing printed by assert itself


In [11]:
bad = pd.DataFrame({"v": [1, 2, 3]}, index=[10, 30, 20])
try:
    assert bad.index.is_monotonic_increasing, "index is not sorted"
except AssertionError as e:
    print("AssertionError:", e)

AssertionError: index is not sorted


The most valuable assertion in research code: row count before and after a merge.

In [12]:
left = pd.DataFrame({"id": ["a", "b", "c"], "v": [1, 2, 3]})
right = pd.DataFrame({"id": ["a", "b", "b"], "w": [10, 20, 21]})   # 'b' appears twice
print(left)
print()
print(right)

  id  v
0  a  1
1  b  2
2  c  3

  id   w
0  a  10
1  b  20
2  b  21


In [13]:
n_before = len(left)
merged = left.merge(right, on="id", how="left")
print("rows before:", n_before, "| after:", len(merged))
merged

rows before: 3 | after: 4


,id,v,w
0,a,1,10.0
1,b,2,20.0
2,b,2,21.0
3,c,3,NaN


In [14]:
try:
    assert len(merged) == n_before, f"merge changed row count {n_before} -> {len(merged)}"
except AssertionError as e:
    print("AssertionError:", e)

AssertionError: merge changed row count 3 -> 4


`b` matched two rows on the right, so the left frame grew from 3 to 4 rows. `validate=`
makes pandas raise instead of you noticing later.

In [15]:
try:
    left.merge(right, on="id", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", str(e)[:80])

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge


## 3. `pd.testing` and `np.testing`: compare two objects

`assert_series_equal` tells you *where* two Series differ. Plain `==` would just give
you a column of True/False.

In [16]:
a = pd.Series([1.0, 2.0, 3.0], index=["x", "y", "z"])
b = pd.Series([1.0, 2.5, 3.0], index=["x", "y", "z"])
try:
    pd.testing.assert_series_equal(a, b)
except AssertionError as e:
    print(e)

Series are different

Series values are different (33.33333 %)
[index]: [x, y, z]
[left]:  [1.0, 2.0, 3.0]
[right]: [1.0, 2.5, 3.0]
At positional index 1, first diff: 2.0 != 2.5


The message says which positions differ (`[1]`), and shows both values.

`np.testing.assert_allclose` compares with a tolerance, for floating point noise.

In [17]:
c = pd.Series([1.0, 2.0000001, 3.0], index=["x", "y", "z"])
np.testing.assert_allclose(a.values, c.values, atol=1e-6)
print("a and c are equal within 1e-6")

a and c are equal within 1e-6


## 4. Where do two frames differ?

`df.compare` shows only the cells that differ, side by side.

In [18]:
x = pd.DataFrame({"p": [1, 2, 3], "q": [10, 20, 30]}, index=["r1", "r2", "r3"])
y = x.copy()
y.loc["r2", "q"] = 99
print(x)
print()
print(y)

    p   q
r1  1  10
r2  2  20
r3  3  30

    p   q
r1  1  10
r2  2  99
r3  3  30


In [19]:
x.compare(y)

q      
    self other
r2  20.0  99.0

Only row `r2`, column `q` differs: 20 in `x` ("self"), 99 in `y` ("other").

To find which *keys* are in one frame but not the other, merge with `indicator=True`.

In [20]:
k1 = pd.DataFrame({"id": ["a", "b", "c"]})
k2 = pd.DataFrame({"id": ["b", "c", "d"]})
k1.merge(k2, on="id", how="outer", indicator=True)

,id,_merge
0,a,left_only
1,b,both
2,c,both
3,d,right_only


`a` is only on the left, `d` only on the right. The same thing for two indexes:

In [21]:
i1 = pd.Index(["a", "b", "c"])
i2 = pd.Index(["b", "c", "d"])
print("in one but not the other:", i1.symmetric_difference(i2).tolist())

in one but not the other: ['a', 'd']


## 5. Recompute a feature two ways

When a vectorised expression is suspicious, recompute it with a tiny loop on a few rows.
The loop is slow but hard to get wrong.

In [22]:
s = pd.Series([10, 20, 30, 40, 50], index=["t1", "t2", "t3", "t4", "t5"])
vec = s.shift(1).rolling(2).mean()
pd.DataFrame({"s": s, "vectorised": vec})

,s,vectorised
t1,10,NaN
t2,20,NaN
t3,30,15.0
t4,40,25.0
t5,50,35.0


By hand: at `t3` the value should be the mean of the two rows before it, (10 + 20) / 2 = 15.

In [23]:
pos = 2                                # position of t3
by_hand = s.iloc[pos - 2:pos].mean()   # rows t1, t2
print("by hand at t3:", by_hand, "| vectorised:", vec.iloc[pos])

by hand at t3: 15.0 | vectorised: 15.0


In [24]:
pos = 4                                # t5: mean of t3, t4 = (30 + 40) / 2
print("by hand at t5:", s.iloc[pos - 2:pos].mean(), "| vectorised:", vec.iloc[pos])

by hand at t5: 35.0 | vectorised: 35.0


## 6. Inspecting around a known bad timestamp

Aggregates hide problems. A slice of a few rows around the suspect time shows the shape of
the problem: gap, duplicate, step change.

In [25]:
ts = pd.Series([1, 2, 3, 5, 6],
               index=pd.to_datetime(["01:00", "02:00", "03:00", "05:00", "06:00"], format="%H:%M"))
ts

1900-01-01 01:00:00    1
1900-01-01 02:00:00    2
1900-01-01 03:00:00    3
1900-01-01 05:00:00    5
1900-01-01 06:00:00    6
dtype: int64

In [26]:
steps = ts.index.to_series().diff()
steps

1900-01-01 01:00:00               NaT
1900-01-01 02:00:00   0 days 01:00:00
1900-01-01 03:00:00   0 days 01:00:00
1900-01-01 05:00:00   0 days 02:00:00
1900-01-01 06:00:00   0 days 01:00:00
dtype: timedelta64[ns]

In [27]:
steps[steps > pd.Timedelta("1h")]

1900-01-01 05:00:00   0 days 02:00:00
dtype: timedelta64[ns]

One step of 2 hours: the hour 04:00 is missing.

On real data, slice a window around a day we suspect.

In [28]:
t = pd.Timestamp("2022-03-27 12:00", tz="UTC")
window = out.loc[t - pd.Timedelta("30h"): t + pd.Timedelta("30h"), ["consumption_mwh"]]
print("rows in a 60h window (expect 61):", len(window))
gaps = window.index.to_series().diff()
gaps[gaps > pd.Timedelta("1h")]

rows in a 60h window (expect 61): 37


time
2022-03-28 00:00:00+00:00   1 days 01:00:00
Name: time, dtype: timedelta64[ns]

A single jump of 25 hours: the whole day 2022-03-27 is absent from the raw file.

## 7. Leakage tests

A 10-row toy. `target` is tomorrow's value. `honest` is yesterday's value. `leaked` is the
target itself with a little noise (someone accidentally built a feature from the future).

In [29]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

toy = pd.DataFrame({"value": [10, 30, 15, 25, 5, 35, 20, 10, 30, 15]},
                   index=["d1", "d2", "d3", "d4", "d5", "d6", "d7", "d8", "d9", "d10"])
toy["target"] = toy["value"].shift(-1)          # tomorrow
toy["honest"] = toy["value"].shift(1)           # yesterday
toy["leaked"] = toy["target"] + 0.1             # tomorrow, with a tiny disguise
toy

,value,target,honest,leaked
d1,10,30.0,NaN,30.1
d2,30,15.0,10.0,15.1
d3,15,25.0,30.0,25.1
d4,25,5.0,15.0,5.1
d5,5,35.0,25.0,35.1
d6,35,20.0,5.0,20.1
d7,20,10.0,35.0,10.1
d8,10,30.0,20.0,30.1
d9,30,15.0,10.0,15.1
d10,15,NaN,30.0,NaN


In [30]:
toy = toy.dropna()
toy

,value,target,honest,leaked
d2,30,15.0,10.0,15.1
d3,15,25.0,30.0,25.1
d4,25,5.0,15.0,5.1
d5,5,35.0,25.0,35.1
d6,35,20.0,5.0,20.1
d7,20,10.0,35.0,10.1
d8,10,30.0,20.0,30.1
d9,30,15.0,10.0,15.1


**Test 1: correlation with the target.** A feature correlated 0.99+ with the target is
the target.

In [31]:
toy[["honest", "leaked"]].corrwith(toy["target"]).round(3)

honest    0.191
leaked    1.000
dtype: float64

**Test 2: lead-lag.** Correlate the feature with `value` shifted by k. An honest feature
correlates best with the present or the past (k ≤ 0). A leaked one correlates best with
the future (k > 0).

In [32]:
for k in [-1, 0, 1]:
    print(f"k={k:+d}  honest: {toy['honest'].corr(toy['value'].shift(-k)):.2f}   leaked: {toy['leaked'].corr(toy['value'].shift(-k)):.2f}")

k=-1  honest: 1.00   leaked: 0.14
k=+0  honest: -0.71   leaked: -0.69
k=+1  honest: 0.14   leaked: 1.00


`honest` peaks at k = −1 (yesterday), `leaked` at k = +1 (tomorrow).

**Test 3: shuffle the target.** Fit on a shuffled target. R² should drop to about zero
for any feature. If it does not, the fitting code itself is broken.

In [33]:
rng = np.random.default_rng(0)
shuffled_target = rng.permutation(toy["target"].values)
m = LinearRegression().fit(toy[["leaked"]], shuffled_target)
print("R² on shuffled target:", round(r2_score(shuffled_target, m.predict(toy[["leaked"]])), 3))

R² on shuffled target: 0.029


### On real data

The same three tests on hourly consumption with a one-hour horizon.

In [34]:
d = clean.set_index("time")[["consumption_mwh"]].copy()
d["target"] = d["consumption_mwh"].shift(-1)                                # next hour
d["honest"] = d["consumption_mwh"].shift(1).rolling(24).mean()              # past only
d["leaked"] = d["consumption_mwh"].shift(-1).rolling(3, center=True).mean() # uses t+1, t+2
d = d.dropna()
split = int(len(d) * 0.8)
tr, te = d.iloc[:split], d.iloc[split:]

In [35]:
for f in ["honest", "leaked", "consumption_mwh"]:
    m = LinearRegression().fit(tr[[f]], tr["target"])
    r2 = r2_score(te["target"], m.predict(te[[f]]))
    best_k = max([-2, -1, 0, 1, 2], key=lambda k: te[f].corr(te["consumption_mwh"].shift(-k)))
    print(f"{f:16s} test R² {r2:.3f}   best lead k = {best_k:+d}")

honest           test R² 0.140   best lead k = -2
leaked           test R² 0.991   best lead k = +1
consumption_mwh  test R² 0.855   best lead k = +0


`consumption_mwh` (the current value) is the persistence benchmark, best lead 0.
`leaked` beats it and its best lead is +1: it knows the future.

## 8. Silent failures

Things that do not raise. Each shown on a 3-row frame.

**Numbers stored as text.** `mean()` fails or, worse, the column is silently dropped by
`numeric_only` aggregations.

In [36]:
f = pd.DataFrame({"price": ["10.5", "11.0", "missing"]})
print(f.dtypes)
print("numeric after coerce:", pd.to_numeric(f["price"], errors="coerce").tolist())

price    object
dtype: object
numeric after coerce: [10.5, 11.0, nan]


**An all-NaN column after a merge** means the key never matched.

In [37]:
l = pd.DataFrame({"id": [1, 2, 3]})
r = pd.DataFrame({"id": ["1", "2", "3"], "w": [10, 20, 30]})     # strings!
try:
    l.merge(r, on="id", how="left")
except ValueError as e:
    print("ValueError:", str(e)[:70])

ValueError: You are trying to merge on int64 and object columns for key 'id'. If y


pandas 2.x raises on int-vs-string keys. With other mismatches (e.g. float vs int, or
string with whitespace) you get NaN instead: check `isna().all()` on the merged columns.

**`inplace=True` returns None.**

In [38]:
g = pd.DataFrame({"v": [1.0, np.nan, 3.0]})
result = g.dropna(inplace=True)
print("returned:", result)
print("g now has", len(g), "rows")

returned: None
g now has 2 rows


**Chained assignment does nothing to the original.**

In [39]:
h = pd.DataFrame({"v": [1, 2, 3]})
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    h[h["v"] > 1]["v"] = 0          # writes into a temporary copy
h

,v
0,1
1,2
2,3


In [40]:
h.loc[h["v"] > 1, "v"] = 0          # the right way
h

,v
0,1
1,0
2,0


**A constant column, a -999 sentinel.** `nunique()` and `min()` show them.

In [41]:
s = pd.DataFrame({"region": ["GB", "GB", "GB"], "temp": [5.0, -999.0, 7.0]})
print("nunique:", s.nunique().to_dict())
print("min    :", s["temp"].min())

nunique: {'region': 1, 'temp': 3}
min    : -999.0


On real data, the same checks in a loop over columns:

In [42]:
for col in raw.columns:
    col_s = raw[col]
    notes = []
    if col_s.dtype == object and pd.to_numeric(col_s, errors="coerce").notna().mean() > 0.9:
        notes.append("object but ~numeric")
    if col_s.nunique() == 1:
        notes.append("constant")
    if pd.api.types.is_numeric_dtype(col_s) and (col_s == -999).any():
        notes.append("-999 sentinel")
    if notes:
        print(f"{col:16s} {notes}")

temp_c           ['-999 sentinel']
price_eur_mwh    ['object but ~numeric']
region           ['constant']


## 9. Warnings as errors, copy-on-write

A warning scrolls past. Turn it into an error while debugging so the traceback points at
the line.

In [43]:
with warnings.catch_warnings():
    warnings.simplefilter("error")
    try:
        pd.to_datetime(pd.Series(["01/02/2023", "13/02/2023"]))   # day-first ambiguity
    except Exception as e:
        print(type(e).__name__, "->", str(e)[:70])

ValueError -> time data "13/02/2023" doesn't match format "%m/%d/%Y", at position 1.


Copy-on-write mode removes the copy/view ambiguity: a subset never writes back to its
parent.

In [44]:
pd.options.mode.copy_on_write = True
p = pd.DataFrame({"v": [1, 2, 3]})
sub = p[p["v"] > 1]
sub.loc[:, "v"] = 0
print("parent untouched:", p["v"].tolist())
pd.options.mode.copy_on_write = False

parent untouched: [1, 2, 3]


Other tools for a live session (not runnable in a batch build):
- `%debug` right after an exception opens the debugger at the failing line.
- `%xmode Verbose` shows local variables in the traceback.
- never `except: pass` in research code; log and `raise`.

## 10. The minimal reproducible example

Shrink until the bug still shows. Eight rows are enough to see the classic misalignment:
`dropna` on X and on y separately.

In [45]:
v = pd.Series([10, 20, 30, 40], index=["t1", "t2", "t3", "t4"])
X = v.to_frame("now")
y = v.shift(-1).rename("next")
print(pd.concat([X, y], axis=1))

    now  next
t1   10  20.0
t2   20  30.0
t3   30  40.0
t4   40   NaN


In [46]:
Xd = X.dropna()      # 4 rows: nothing to drop
yd = y.dropna()      # 3 rows: t4 is gone
print("len X:", len(Xd), "| len y:", len(yd))

len X: 4 | len y: 3


In [47]:
pd.DataFrame({"X_row": Xd.index[:3], "y_row": yd.index[:3]})

,X_row,y_row
0,t1,t1
1,t2,t2
2,t3,t3


Pairing by position puts `t1`'s features against `t1`'s label, fine, but the lengths
differ and any later `.values[:n]` truncation silently pairs the wrong rows. The fix:
one frame, one `dropna`.

In [48]:
frame = pd.concat([X, y], axis=1).dropna()
frame

,now,next
t1,10,20.0
t2,20,30.0
t3,30,40.0


## 11. The pandas errors you will meet

Read a traceback from the bottom. Each error below is reproduced on 2–3 rows with its
usual cause.

In [49]:
s1 = pd.Series([1, 2], index=["a", "b"])
s2 = pd.Series([1, 2], index=["b", "c"])
try:
    s1 == s2
except ValueError as e:
    print("Series with different labels compared ->", e)

Series with different labels compared -> Can only compare identically-labeled Series objects


In [50]:
dup = pd.Series([1, 2, 3], index=[0, 0, 1])
try:
    dup.reindex([0, 1, 2])
except ValueError as e:
    print("duplicate labels ->", e)

duplicate labels -> cannot reindex on an axis with duplicate labels


In [51]:
strings = pd.Series(["2023-01-01", "2023-01-02"])
try:
    strings.dt.hour
except AttributeError as e:
    print("strings not datetimes ->", e)

strings not datetimes -> Can only use .dt accessor with datetimelike values


In [52]:
try:
    if pd.Series([True, False]):
        pass
except ValueError as e:
    print("Series in if ->", str(e)[:60])

Series in if -> The truth value of a Series is ambiguous. Use a.empty, a.boo


In [53]:
try:
    pd.Timestamp("2023-01-01") < pd.Timestamp("2023-01-01", tz="UTC")
except TypeError as e:
    print("naive vs aware ->", e)

naive vs aware -> Cannot compare tz-naive and tz-aware timestamps


In [54]:
a1 = pd.DataFrame({"id": [1, 2], "v": [1, 2]})
both = a1.merge(a1, on="id")      # v appears twice -> v_x, v_y
print(both.columns.tolist())
try:
    both["v"]
except KeyError as e:
    print("KeyError after merge ->", e)

['id', 'v_x', 'v_y']
KeyError after merge -> 'v'


## 12. "R² is 0.99, is that fine?" The six checks

A model with one leaked feature, then the checks in order.

In [55]:
d = clean.set_index("time")[["consumption_mwh", "temp_c"]].copy()
d["target"] = d["consumption_mwh"].shift(-24)                     # 24h ahead
d["lag0"] = d["consumption_mwh"]                                  # now
d["lag168"] = d["consumption_mwh"].shift(144)                     # same hour last week, relative to the target
d["roll_c"] = d["consumption_mwh"].shift(-25).rolling(3).mean()   # centred on the TARGET hour -> leak
d = d.dropna()
feats = ["lag0", "lag168", "roll_c", "temp_c"]
split = int(len(d) * 0.8)
tr, te = d.iloc[:split], d.iloc[split:]
m = LinearRegression().fit(tr[feats], tr["target"])
print("reported test R²:", round(r2_score(te["target"], m.predict(te[feats])), 4))

reported test R²: 0.9912


**Check 1: what is one row and what is the target?** Rows are hours, target is consumption 24 h later.

**Check 2: is the split chronological?**

In [56]:
print("train ends:", tr.index[-1], "| test starts:", te.index[0])

train ends: 2023-08-08 07:00:00+00:00 | test starts: 2023-08-08 08:00:00+00:00


**Check 3: how much better than the naive baseline?**

In [57]:
print("naive (same hour yesterday) R²:", round(r2_score(te["target"], te["lag0"]), 3))
print("model R²                      :", round(r2_score(te["target"], m.predict(te[feats])), 3))

naive (same hour yesterday) R²: 0.828
model R²                      : 0.991


**Check 4: coefficients.** One feature near 1 and the rest near 0 is a red flag.

In [58]:
pd.Series(m.coef_, index=feats).round(3)

lag0      0.024
lag168    0.026
roll_c    0.991
temp_c    8.458
dtype: float64

**Check 5: correlation of each feature with the target.**

In [59]:
te[feats].corrwith(te["target"]).round(3)

lag0      0.914
lag168    0.936
roll_c    0.995
temp_c   -0.000
dtype: float64

**Check 6: drop the suspect and refit.**

In [60]:
feats_ok = ["lag0", "lag168", "temp_c"]
m2 = LinearRegression().fit(tr[feats_ok], tr["target"])
print("R² without roll_c:", round(r2_score(te["target"], m2.predict(te[feats_ok])), 3))

R² without roll_c: 0.897


The R² collapses when one feature is removed, that feature has the largest coefficient
and the highest correlation with the target. Read its definition: a 3-hour mean centred
on the target hour. That is the leak.

## Quick reference

| Situation | Do this |
|---|---|
| any surprise | restart kernel, run all, fix seeds |
| pipeline of steps | `.pipe(trace, "label")` between steps |
| merge | row count before/after, `validate=`, `indicator=True` |
| "is this feature right?" | recompute 2 rows by hand |
| two frames should match | `df.compare`, `pd.testing.assert_frame_equal` |
| known bad timestamp | slice around it, `diff()` of the index |
| result too good | six checks: row/target, split, baseline, coefficients, corr, drop-and-refit |
| leaked feature suspected | corr with target, lead-lag, shuffled target |
| warnings scroll past | `warnings.simplefilter("error")` |
| copy/view confusion | `pd.options.mode.copy_on_write = True` |
| exception | `%debug`, read the traceback bottom-up |
| big data, small bug | shrink to a few rows that still reproduce |